<a href="https://colab.research.google.com/github/BalakeerthiNidumolu/AyurBot/blob/main/Ayur_Bot.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import tensorflow as tf
import json
from tensorflow.keras.preprocessing.image import ImageDataGenerator
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Conv2D, MaxPooling2D, Flatten, Dense, Dropout
from tqdm.keras import TqdmCallback

IMG_SIZE = 224
DATASET_PATH = "40"

datagen = ImageDataGenerator(
    rescale=1./255,
    validation_split=0.2
)

train = datagen.flow_from_directory(
    DATASET_PATH,
    target_size=(IMG_SIZE, IMG_SIZE),
    class_mode="categorical",
    subset="training"
)

val = datagen.flow_from_directory(
    DATASET_PATH,
    target_size=(IMG_SIZE, IMG_SIZE),
    class_mode="categorical",
    subset="validation"
)

with open("class_labels.json", "w") as f:
    json.dump(train.class_indices, f)

model = Sequential([
    Conv2D(32, (3,3), activation="relu", input_shape=(IMG_SIZE, IMG_SIZE, 3)),
    MaxPooling2D(),
    Conv2D(64, (3,3), activation="relu"),
    MaxPooling2D(),
    Flatten(),
    Dense(128, activation="relu"),
    Dropout(0.5),
    Dense(train.num_classes, activation="softmax")
])

model.compile(
    optimizer="adam",
    loss="categorical_crossentropy",
    metrics=["accuracy"]
)

model.fit(train, validation_data=val, epochs=15, callbacks=[TqdmCallback()])
model.save("plant_cnn.h5")


In [ ]:
import tensorflow as tf
import numpy as np
import cv2
import json
import os

# ================= CONFIG =================
MODEL_PATH = "plant_cnn.h5"
LABEL_PATH = "class_labels.json"
IMG_SIZE = 224

# ================= LOAD MODEL =================
model = tf.keras.models.load_model(MODEL_PATH)

# ================= LOAD LABELS =================
with open(LABEL_PATH, "r") as f:
    class_indices = json.load(f)

# Reverse mapping: index -> class name
labels = {v: k for k, v in class_indices.items()}

# ================= PREDICTION FUNCTION =================
def predict_plant(image_path):
    # ---- Check path ----
    if not os.path.exists(image_path):
        return "Image path not found", 0.0

    # ---- Read image ----
    img = cv2.imread(image_path)

    if img is None:
        return "Invalid or corrupted image", 0.0

    # ---- Preprocess ----
    img = cv2.resize(img, (IMG_SIZE, IMG_SIZE))
    img = img.astype("float32") / 255.0
    img = np.expand_dims(img, axis=0)

    # ---- Predict ----
    preds = model.predict(img, verbose=0)
    idx = np.argmax(preds)
    confidence = float(preds[0][idx] * 100)

    return labels[idx], confidence

# ================= TEST =================
if __name__ == "__main__":
    image_path = r"D:\petty money\Plant\40\Aloe Vera\2.jpg"
    plant, conf = predict_plant(image_path)

    print("🌱 Predicted Plant :", plant)
    print("✅ Confidence      :", round(conf, 2), "%")
